# SILVA Physics-Informed Equilibrium

This lab derives a physics-informed equilibrium for an ODE initial-value
problem, computes time derivatives with the implicit function theorem, and
trains a tiny linear-decay task with boundary, equation, and Jacobian terms.
The construction follows Physics-Informed Deep Equilibrium Models [51].

<!-- silva-numbered-citations:start -->
**Numbered literature:** [1](https://jseluis.github.io/silva-networks/paper/references/#ref-1), [4](https://jseluis.github.io/silva-networks/paper/references/#ref-4), [6](https://jseluis.github.io/silva-networks/paper/references/#ref-6), [14](https://jseluis.github.io/silva-networks/paper/references/#ref-14), [51](https://jseluis.github.io/silva-networks/paper/references/#ref-51). Each number opens the complete citation and its primary external source.
<!-- silva-numbered-citations:end -->


In [ ]:
from pathlib import Path
import importlib.util
import subprocess
import sys

REPO_URL = "https://github.com/jseluis/silva-networks.git"

def find_local_silva_root():
    candidates = [Path.cwd(), Path("/content/silva-networks")]
    root = Path.cwd()
    while root != root.parent:
        candidates.append(root)
        root = root.parent
    for candidate in candidates:
        if (candidate / "src" / "silva_networks").exists():
            return candidate
    return None

root = find_local_silva_root()
if root is not None:
    sys.path.insert(0, str(root / "src"))
elif importlib.util.find_spec("silva_networks") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", f"git+{REPO_URL}"])
    root = Path.cwd()
else:
    root = Path.cwd()

In [ ]:
import torch
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 300, "savefig.dpi": 300})

from silva_networks import (
    SILVAPhysicsInformedEquilibrium,
    SolverConfig,
    make_linear_ivp_dataset,
)

torch.manual_seed(240)

## 1. Initial-Value Problem

Consider

$$
\frac{dy}{dt}=N(t,y(t)),
\qquad y(t_0)=y_0.
$$

A physics-informed model is trained at collocation times without requiring a
target state at every point. The package's analytic batch supplies targets only
so this notebook can measure error after training.

In [ ]:
data = make_linear_ivp_dataset(points=9, final_time=1.5, rate=-0.5)
assert data.equation_residual().abs().max() == 0
assert torch.allclose(data.target[:1], data.initial_state)
print("collocation points:", data.times.shape[0])
print("exact final state:", float(data.target[-1]))

## 2. State Is Defined Implicitly

Instead of evaluating a finite stack, define

$$
z^\star(t)=f_\theta(z^\star(t),t),
\qquad \widehat y(t)=Q_\psi(z^\star(t)).
$$

In SILVA, time enters through the source branch and the latent state enters
through the self-interaction branch. A standard output loss can use the
package's implicit adjoint, avoiding storage of every forward solver iterate.

## 3. Time Derivative from the Implicit Function Theorem

Differentiate the fixed-point equation:

$$
\frac{dz^\star}{dt}
=J_zf_\theta\frac{dz^\star}{dt}+J_tf_\theta.
$$

Therefore

$$
\frac{dz^\star}{dt}
=(I-J_zf_\theta)^{-1}J_tf_\theta,
\qquad
\frac{d\widehat y}{dt}=J_Q\frac{dz^\star}{dt}.
$$

`implicit_time_derivative` solves this system with either a dense latent
Jacobian or matrix-free JVPs and GMRES. `auto` uses the dense path only below
the configured latent-dimension threshold. Both paths differentiate the same
implicit equation.

In [ ]:
model = SILVAPhysicsInformedEquilibrium(
    state_dim=4,
    output_dim=1,
    state_scale=0.15,
    config=SolverConfig(
        solver="picard",
        max_iter=15,
        tol=1e-6,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
initial = model(data.times, return_result=True)
derivative = model.implicit_time_derivative(data.times, initial.state, mode="dense")
matrix_free_derivative = model.implicit_time_derivative(
    data.times,
    initial.state,
    mode="matrix_free",
)
assert derivative.shape == initial.output.shape == data.target.shape
torch.testing.assert_close(matrix_free_derivative, derivative, atol=1e-5, rtol=1e-5)
print("equilibrium residual:", initial.solver_result.residual)

## 4. Three-Term Physics-Informed Objective

The decomposed objective is

$$
\mathcal J_b=\|\widehat y(t_0)-y_0\|_2^2,
$$

$$
\mathcal J_N
=\frac1M\sum_{i=1}^M
\left\|\frac{d\widehat y(t_i)}{dt}
-N(t_i,\widehat y(t_i))\right\|_2^2,
$$

$$
\mathcal J
=\mathcal J_b+\lambda\mathcal J_N
+\kappa\|J_zf_\theta\|_F^2.
$$

The Jacobian term is estimated with Rademacher probes [6, 14, 51]. It is a
solver-conditioning term, not the differential-equation residual.

In [ ]:
loss = model.physics_loss(
    data.times,
    data.dynamics,
    initial_time=data.times[:1],
    initial_state=data.initial_state,
    physics_weight=1.0,
    jacobian_weight=1e-3,
    jacobian_samples=1,
)
print("boundary:", float(loss.initial))
print("ODE residual:", float(loss.residual))
print("Jacobian estimate:", float(loss.jacobian))

## 5. Tiny Physics-Only Training Run

No trajectory targets appear in the optimization objective below. They are
used afterward only to calculate an error curve. This separation is essential
when describing a physics-informed experiment.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
history = []
for _ in range(8):
    optimizer.zero_grad()
    terms = model.physics_loss(
        data.times,
        data.dynamics,
        initial_time=data.times[:1],
        initial_state=data.initial_state,
        jacobian_weight=1e-3,
    )
    terms.total.backward()
    optimizer.step()
    history.append((float(terms.initial.detach()), float(terms.residual.detach())))

trained = model(data.times, return_result=True)
trajectory_mse = torch.nn.functional.mse_loss(trained.output, data.target)
print("trajectory MSE used for evaluation:", float(trajectory_mse))

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(7.0, 2.6))
axes[0].plot(data.times[:, 0], data.target[:, 0], label="exact", linewidth=2)
axes[0].plot(data.times[:, 0], trained.output.detach()[:, 0], "--", label="SILVA")
axes[0].set(xlabel="time", ylabel="state")
axes[0].legend()
axes[1].semilogy([item[0] for item in history], label="boundary")
axes[1].semilogy([item[1] for item in history], label="ODE residual")
axes[1].set(xlabel="optimization step", ylabel="loss")
axes[1].legend()
figure.tight_layout()
plt.show()

## 6. Stiff Systems and Scaling

An equilibrium layer does not automatically solve stiffness. Stability depends
on the transition, root solver, Jacobian spectrum, collocation distribution,
and loss balancing. For larger states, use matrix-free implicit products,
report forward and backward tolerances separately, and compare against a
trusted numerical integrator on the same time interval.

## From 24 Silva Physics Informed Equilibrium to a Custom SILVA Family

The construction in this notebook can be separated into the universal
conditioned-equilibrium contract

$$
z_0=I_\eta(x),\qquad
z^\star=T_\theta(z^\star,x),\qquad
\widehat y=Q_\psi(z^\star).
$$

For this topic:

| Part | Concrete interpretation |
| --- | --- |
| Equilibrium state | an implicit latent state or coupled differential/algebraic stage state |
| Condition | time, initial/boundary values, dynamics, and algebraic constraints |
| Repeated computation | a time-conditioned fixed point or implicit Runge-Kutta root map |
| Required invariants | initial/boundary conditions, equation shape, and constraint consistency |
| Replaceable components | time/source lift, transition, readout, dynamics, constraints, losses, and solvers |

The initializer and source path are evaluated outside or alongside the root
solve. Only the state-preserving transition is repeated. Replacing an internal
architecture does not change this equation, provided the transition still maps
the same state space into itself.


### Build the Physics-Informed Transition Explicitly

```python
class MyPhysicsTransition(nn.Module):
    time_dim = 1
    state_dim = 64

    def __init__(self):
        super().__init__()
        self.time_source = MyTimeEncoder()
        self.state_operator = MyResidualOrOperatorNetwork()

    def forward(self, state, times):
        source = self.time_source(times)
        return torch.tanh(source + 0.2 * self.state_operator(state))


model = SILVAPhysicsInformedEquilibrium(
    state_dim=64,
    output_dim=physical_dimension,
    transition=MyPhysicsTransition(),
    readout=my_physical_readout,
    derivative_mode="matrix_free",
    derivative_max_iter=100,
    derivative_tol=1e-7,
    config=solver_config,
)
```

`physics_loss` then accepts the user-supplied dynamics. The transition defines
the implicit representation; the dynamics define the differential-equation
residual. They are related by the implicit time derivative but are not the same
module.


In [ ]:
import torch as silva_extension_torch
from torch import nn as silva_extension_nn

from silva_networks import (
    SILVAConditionedEquilibrium,
    SILVAZeroInitializer,
    SolverConfig,
    validate_silva_transition,
)


class NotebookExtensionTransition(silva_extension_nn.Module):
    def __init__(self, condition_dim=2, state_dim=3):
        super().__init__()
        self.source = silva_extension_nn.Linear(condition_dim, state_dim)
        self.state_field = silva_extension_nn.Sequential(
            silva_extension_nn.Linear(state_dim, 2 * state_dim),
            silva_extension_nn.Tanh(),
            silva_extension_nn.Linear(2 * state_dim, state_dim),
        )

    def forward(self, state, condition):
        return silva_extension_torch.tanh(
            self.source(condition) + 0.15 * self.state_field(state)
        )


silva_extension_torch.manual_seed(610)
notebook_condition = silva_extension_torch.linspace(-1.0, 1.0, 8).reshape(4, 2)
notebook_state0 = silva_extension_torch.zeros(4, 3)
notebook_transition = NotebookExtensionTransition()

notebook_report = validate_silva_transition(
    notebook_transition,
    notebook_state0,
    notebook_condition,
)
assert notebook_report.valid

with silva_extension_torch.no_grad():
    notebook_reference_step = silva_extension_torch.tanh(
        notebook_transition.source(notebook_condition)
        + 0.15 * notebook_transition.state_field(notebook_state0)
    )
silva_extension_torch.testing.assert_close(
    notebook_transition(notebook_state0, notebook_condition),
    notebook_reference_step,
)

notebook_custom_model = SILVAConditionedEquilibrium(
    notebook_transition,
    SILVAZeroInitializer(3),
    readout=silva_extension_nn.Linear(3, 1),
    config=SolverConfig(
        solver="picard",
        max_iter=40,
        tol=1e-7,
        backward_mode="implicit",
        backward_solver="gmres",
        anderson_batch_dims=1,
    ),
)
notebook_custom_result = notebook_custom_model(
    notebook_condition,
    return_result=True,
)
assert notebook_custom_result.output.shape == (4, 1)
assert notebook_custom_result.solver_result.residual < 1e-5

notebook_custom_result.output.square().mean().backward()
assert all(
    parameter.grad is not None and silva_extension_torch.isfinite(parameter.grad).all()
    for parameter in notebook_custom_model.parameters()
)
print("custom transition:", notebook_report)
print("equilibrium residual:", notebook_custom_result.solver_result.residual)


## Numerical Equivalence, Compact Reproduction, and Scale

Before training, compare one packaged transition with an independently written
update:

$$
e_{\mathrm{step}}
=\frac{\|T_\theta(z,x)-T_{\mathrm{ref}}(z,x)\|_2}
{\|T_{\mathrm{ref}}(z,x)\|_2+\varepsilon}.
$$

After solving, report the fixed-point residual separately:

$$
e_{\mathrm{fp}}
=\frac{\|T_\theta(z^\star,x)-z^\star\|_2}
{\|z^\star\|_2+\varepsilon}.
$$

For this notebook, a compact reproduction must declare and assert
**equation residual, boundary error, trajectory error, and solver residual**. A full experiment must additionally record the
source dataset version and split, preprocessing, architecture widths, solver
and optimizer schedules, random seeds, baseline configuration, checkpoints,
and every deviation from the cited protocol.

The principal scaling axes are **collocation count, latent dimension, stages, stiffness, and time horizon**. Increase one axis at
a time, retain the compact deterministic case as a regression test, and record
task error, domain-specific residual, forward residual, backward linear
residual, memory use, and runtime independently.

### Extension Exercises

1. Replace one component from this notebook while preserving its state and
   domain invariants.
2. Write the replacement first as an independent reference function, then as
   a module, and assert one-step equivalence.
3. Compare two solver configurations on the identical trained transition.
4. Add a compact baseline and a predeclared metric threshold.
5. Create a full-scale configuration without weakening the compact tests.

The complete authoring protocol is documented in
[Extending SILVA](https://jseluis.github.io/silva-networks/learn/extending-silva/).


In [ ]:
notebook_reproduction_record = {
    "notebook": '24_silva_physics_informed_equilibrium.ipynb',
    "state": 'an implicit latent state or coupled differential/algebraic stage state',
    "condition": 'time, initial/boundary values, dynamics, and algebraic constraints',
    "transition": 'a time-conditioned fixed point or implicit Runge-Kutta root map',
    "invariants": 'initial/boundary conditions, equation shape, and constraint consistency',
    "compact_metric": 'equation residual, boundary error, trajectory error, and solver residual',
    "scale_axis": 'collocation count, latent dimension, stages, stiffness, and time horizon',
}
assert all(notebook_reproduction_record.values())
notebook_reproduction_record


## Where to Go Next

| Question | Page |
| --- | --- |
| How is the implicit time derivative obtained? | [Physics-Informed Equilibria](https://jseluis.github.io/silva-networks/learn/physics-informed-equilibria/#implicit-time-derivative) |
| Which analytic ODE is used? | [Advanced Equilibrium Datasets](https://jseluis.github.io/silva-networks/learn/advanced-equilibrium-datasets/#linear-ode-ivp) |
| Which physics-loss objects are public? | [Physics-Informed API](https://jseluis.github.io/silva-networks/api/physics_informed/) |
